# SCOPE / REACH Outcome Analysis

Reads all outcomes from `generation.tracked_events` in `pipeline_config.yaml`.
Produces per-outcome AUC and calibration plots, and a single combined efficiency figure.

In [ ]:
import pathlib
import yaml

# ── CONFIG ──────────────────────────────────────────────────────────────────
OUTPUT_DIR   = pathlib.Path("scope_reach_output")   # directory produced by run_timelines.py
CONFIG_YAML  = pathlib.Path("pipeline_config.yaml") # read tracked_events from here
MAX_PATIENTS = 100          # max_patients from pipeline_config.yaml (None = full cohort)
SHUFFLE_SEED = 47           # shuffle_seed from pipeline_config.yaml
MARGINAL     = True         # True: REACH cost = avg_tokens_m2 only (M1 already paid)
                            # False: REACH cost = avg_tokens_m1 + avg_tokens_m2 (standalone)
# ────────────────────────────────────────────────────────────────────────────

with open(CONFIG_YAML) as _f:
    _pipeline_cfg = yaml.safe_load(_f)

OUTCOME_NAMES = _pipeline_cfg["generation"]["tracked_events"]
print(f"Outcomes from config ({len(OUTCOME_NAMES)}):")
for _n in OUTCOME_NAMES:
    print(f"  {_n}")

# Validate all required files exist up front
index_path = OUTPUT_DIR / "patient_index.parquet"
if not index_path.exists():
    raise FileNotFoundError(index_path)
for _name in OUTCOME_NAMES:
    _safe = _name.replace("/", "_").replace(" ", "_")
    _p = OUTPUT_DIR / f"scores_{_safe}.npz"
    if not _p.exists():
        raise FileNotFoundError(_p)

In [ ]:
import logging
import typing
import warnings

import joblib as jl
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy.stats import rankdata
from sklearn import metrics as skl_mets
from sklearn.metrics import brier_score_loss

plt.rcParams.update({"text.usetex": False, "font.family": "serif", "mathtext.fontset": "cm"})
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#8B6156", "#b879d1cf", "#00d192", "#d62728"])

COLORS  = {"Monte Carlo": "#8B6156", "SCOPE": "#b879d1cf", "REACH": "#00d192"}
MARKERS = {"Monte Carlo": "o",       "SCOPE": "s",          "REACH": "^"}

## Load data

In [ ]:
import json

run_summary_path = OUTPUT_DIR / "run_summary.json"
if not run_summary_path.exists():
    raise FileNotFoundError(run_summary_path)
with open(run_summary_path) as _f:
    _run_summary = json.load(_f)

_run_subject_ids = _run_summary.get("subject_ids")
if _run_subject_ids is None:
    raise KeyError(
        "run_summary.json has no 'subject_ids' key — re-run run_timelines.py to regenerate."
    )

# Build subject_id → run-local patient_idx mapping (shared across all outcomes)
_sid_map = pl.DataFrame({
    "subject_id": _run_subject_ids,
    "patient_idx": list(range(len(_run_subject_ids))),
})

_index_df_full = (
    pl.read_parquet(index_path)
    .drop("patient_idx")
    .filter(pl.col("subject_id").is_in(_run_subject_ids))
    .join(_sid_map, on="subject_id", how="left")
    .sort("patient_idx")
)

_ALL_SAMPLE_SIZES = [1, 2, 5, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]

outcome_data = {}
for _name in OUTCOME_NAMES:
    _safe = _name.replace("/", "_").replace(" ", "_")
    _scores_path = OUTPUT_DIR / f"scores_{_safe}.npz"

    _outcome_col = f"{_name}_future"
    if _outcome_col not in _index_df_full.columns:
        print(f"WARNING: '{_outcome_col}' not found in patient_index — skipping {_name}")
        continue

    _y_true = _index_df_full[_outcome_col].to_numpy().astype(bool)

    _data   = np.load(_scores_path)
    _M0     = _data["M0"]
    _M1     = _data["M1"]
    _M2     = _data["M2"]
    _arr_m0 = _data["M0_raw"]
    _arr_m1 = _data["M1_raw"]
    _arr_m2 = _data["M2_raw"]
    _lens_m0 = (~np.isnan(_arr_m0)).sum(axis=1).astype(np.int32)
    _lens_m1 = (~np.isnan(_arr_m1)).sum(axis=1).astype(np.int32)
    _lens_m2 = (~np.isnan(_arr_m2)).sum(axis=1).astype(np.int32)

    assert len(_y_true) == len(_M0), (
        f"{_name}: y_true ({len(_y_true)}) vs scores ({len(_M0)}) length mismatch"
    )

    _avg_m1  = float(_data["avg_m1_tokens"][0]) if "avg_m1_tokens" in _data.files else np.nan
    _avg_m2  = float(_data["avg_m2_tokens"][0]) if "avg_m2_tokens" in _data.files else np.nan
    _n_pts   = len(_y_true)
    _n_pos   = int(_y_true.sum())
    _max_n   = max(int(_lens_m1.max()), int(_lens_m2.max())) if _n_pts > 0 else 10
    _ss      = sorted({s for s in _ALL_SAMPLE_SIZES if s <= _max_n} | {_max_n})

    outcome_data[_name] = {
        "y_true":       _y_true,
        "M0": _M0,  "M1": _M1,  "M2": _M2,
        "arr_m0":       _arr_m0,
        "arr_m1":       _arr_m1,
        "arr_m2":       _arr_m2,
        "lens_m0":      _lens_m0,
        "lens_m1":      _lens_m1,
        "lens_m2":      _lens_m2,
        "avg_tokens_m1": _avg_m1,
        "avg_tokens_m2": _avg_m2,
        "n_patients":   _n_pts,
        "n_pos":        _n_pos,
        "prevalence":   float(_y_true.mean()),
        "CAN_COMPUTE_AUC": _n_pos > 0 and _n_pos < _n_pts,
        "SAMPLE_SIZES": _ss,
        "N_STRAPS":     min(500, max(50, _n_pts * 3)),
        "N_BINS":       max(3, min(10, _n_pts // 15)),
        "N_BOOT_CI":    min(2000, max(200, _n_pts * 10)),
    }
    print(
        f"  {_name}: n={_n_pts}, n_pos={_n_pos} ({_y_true.mean():.3f}), "
        f"avg_m1={_avg_m1:.1f} tok, avg_m2={_avg_m2:.1f} tok"
    )

print(f"\nLoaded {len(outcome_data)} outcomes.")

## Bootstrap utilities

In [ ]:
Generator: typing.TypeAlias = np.random._generator.Generator


def bootstrap_ci(
    y_true: np.ndarray,
    y_score: np.ndarray,
    *,
    n_samples: int = 10_000,
    alpha: float = 0.05,
    rng: Generator = np.random.default_rng(seed=42),
    objs: tuple = ("roc_auc", "brier"),
    n_jobs: int = -1,
) -> dict:
    """Percentile bootstrap CI for roc_auc and/or brier score."""

    def _one(rng_i):
        warnings.filterwarnings("ignore")
        idx = rng_i.choice(len(y_true), size=len(y_true), replace=True)
        yt, ys = y_true[idx], y_score[idx]
        out = {}
        if "roc_auc" in objs:
            out["roc_auc"] = skl_mets.roc_auc_score(yt, ys)
        if "brier" in objs:
            out["brier"] = skl_mets.brier_score_loss(yt, ys)
        return out

    with jl.Parallel(n_jobs=n_jobs) as par:
        scores = par(jl.delayed(_one)(ri) for ri in rng.spawn(n_samples))

    return {
        ob: np.nanquantile([s[ob] for s in scores], q=[alpha / 2, 1 - alpha / 2])
        for ob in objs
    }


def bootstrap_pval(
    y_true: np.ndarray,
    y_score0: np.ndarray,
    y_score1: np.ndarray,
    *,
    n_samples: int = 10_000,
    rng: Generator = np.random.default_rng(seed=42),
    alternative: typing.Literal["one-sided", "two-sided"] = "one-sided",
    objs: tuple = ("roc_auc", "brier"),
    n_jobs: int = -1,
) -> dict:
    """Bootstrap p-value: H0 that score0 and score1 are equally good."""

    def _diffs(rng_i):
        warnings.filterwarnings("ignore")
        idx = rng_i.choice(len(y_true), size=len(y_true), replace=True)
        yt0, ys0 = y_true[idx], y_score0[idx]
        yt1, ys1 = y_true[idx], y_score1[idx]
        d = {}
        if "roc_auc" in objs:
            d["roc_auc"] = skl_mets.roc_auc_score(yt1, ys1) - skl_mets.roc_auc_score(yt0, ys0)
        if "brier" in objs:
            d["brier"] = -(skl_mets.brier_score_loss(yt1, ys1) - skl_mets.brier_score_loss(yt0, ys0))
        return d

    obs_diffs = {}
    if "roc_auc" in objs:
        obs_diffs["roc_auc"] = skl_mets.roc_auc_score(y_true, y_score1) - skl_mets.roc_auc_score(y_true, y_score0)
    if "brier" in objs:
        obs_diffs["brier"] = -(skl_mets.brier_score_loss(y_true, y_score1) - skl_mets.brier_score_loss(y_true, y_score0))

    with jl.Parallel(n_jobs=n_jobs) as par:
        boot_diffs = par(jl.delayed(_diffs)(ri) for ri in rng.spawn(n_samples))

    pvals = {}
    for ob in objs:
        d_obs  = obs_diffs[ob]
        d_boot = np.array([b[ob] for b in boot_diffs])
        if alternative == "one-sided":
            pvals[ob] = float(np.mean(d_boot >= d_obs))
        else:
            pvals[ob] = float(np.mean(np.abs(d_boot) >= np.abs(d_obs)))
    return pvals

## Full-sample AUC / Brier with bootstrap CIs

In [ ]:
rng_ci = np.random.default_rng(seed=42)

for _name, _od in outcome_data.items():
    print(f"\n=== {_name} ===")
    if not _od["CAN_COMPUTE_AUC"]:
        print(f"  Skipping — n_pos={_od['n_pos']} in {_od['n_patients']} patients.")
        continue

    _y = _od["y_true"]
    print(f"  {'Estimator':<14}  {'AUC':>6}  {'95% CI':>13}  {'Brier':>7}  {'95% CI':>13}")
    print("  " + "-" * 60)
    for _est_name, _scores in [("Monte Carlo", _od["M0"]), ("SCOPE", _od["M1"]), ("REACH", _od["M2"])]:
        _valid = ~np.isnan(_scores)
        if _valid.sum() == 0:
            print(f"  {_est_name:<14}  no valid scores")
            continue
        _auc   = skl_mets.roc_auc_score(_y[_valid], _scores[_valid])
        _brier = brier_score_loss(_y[_valid], _scores[_valid])
        _ci    = bootstrap_ci(_y[_valid], _scores[_valid], n_samples=_od["N_BOOT_CI"], rng=rng_ci)
        _alo, _ahi = _ci["roc_auc"]
        _blo, _bhi = _ci["brier"]
        print(
            f"  {_est_name:<14}  {_auc:.4f}  [{_alo:.4f},{_ahi:.4f}]"
            f"  {_brier:.5f}  [{_blo:.5f},{_bhi:.5f}]"
        )

    # Empirical spontaneity: mean over patients of p*(1-p), where p = individual REACH score
    _m2 = _od["M2"]
    _valid_m2 = _m2[~np.isnan(_m2)]
    _spontaneity = float(np.mean(_valid_m2 * (1.0 - _valid_m2)))
    print(f"  {'Spontaneity':<14}  {_spontaneity:.5f}  (mean p·(1-p) over {len(_valid_m2)} patients, p=REACH)")

## AUC vs number of samples (subsampling bootstrap)

In [ ]:
def _make_ctx(arr, lens, y_true):
    n_patients, L = arr.shape
    y_true_bool = y_true.astype(bool)
    n_pos = int(y_true_bool.sum())
    n_neg = int(len(y_true) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return None
    return {
        "arr":         arr,
        "lens":        lens,
        "L":           L,
        "n_patients":  n_patients,
        "lens_min":    int(lens.min()),
        "lens_max":    int(lens.max()),
        "invalid":     (np.arange(L)[None, :] >= lens[:, None]),
        "y_true_bool": y_true_bool,
        "n_pos":       n_pos,
        "n_neg":       n_neg,
        "auc_offset":  0.5 * n_pos * (n_pos + 1),
        "auc_norm":    1.0 / (n_pos * n_neg),
        "full_pred":   np.nanmean(arr, axis=1),
    }


def _batched_auc(y_preds, ctx):
    ranks   = rankdata(y_preds, axis=1)
    sum_pos = ranks[:, ctx["y_true_bool"]].sum(axis=1)
    return (sum_pos - ctx["auc_offset"]) * ctx["auc_norm"]


def _subsample_aucs(ctx, n_samps, n_straps, rng):
    arr        = ctx["arr"]
    L          = ctx["L"]
    n_patients = ctx["n_patients"]
    invalid    = ctx["invalid"]

    if n_samps >= ctx["lens_max"]:
        y_pred  = ctx["full_pred"]
        y_preds = np.broadcast_to(y_pred, (n_straps, n_patients))
        return _batched_auc(y_preds, ctx)

    noise  = rng.random((n_straps, n_patients, L), dtype=np.float32)
    noise[:, invalid] = 2.0
    top_k  = np.argpartition(noise, n_samps, axis=2)[:, :, :n_samps]
    y_preds = np.nanmean(
        np.take_along_axis(arr[None], top_k, axis=2), axis=2
    ).astype(np.float32)
    return _batched_auc(y_preds, ctx)


rng_sub = np.random.default_rng(seed=0)

for _name, _od in outcome_data.items():
    if not _od["CAN_COMPUTE_AUC"]:
        _od["auc_list"] = []
        continue

    _ctx_m0 = _make_ctx(_od["arr_m0"], _od["lens_m0"], _od["y_true"])
    _ctx_m1 = _make_ctx(_od["arr_m1"], _od["lens_m1"], _od["y_true"])
    _ctx_m2 = _make_ctx(_od["arr_m2"], _od["lens_m2"], _od["y_true"])

    _auc_list = []
    for _n in _od["SAMPLE_SIZES"]:
        _mc = _subsample_aucs(_ctx_m0, _n, _od["N_STRAPS"], rng_sub)
        _sc = _subsample_aucs(_ctx_m1, _n, _od["N_STRAPS"], rng_sub)
        _re = _subsample_aucs(_ctx_m2, _n, _od["N_STRAPS"], rng_sub)
        for _a, _b, _c in zip(_mc, _sc, _re):
            _auc_list.append(((float(_a), float(_b), float(_c)), _n))
    _od["auc_list"] = _auc_list

print("Bootstrap complete.")
for _name, _od in outcome_data.items():
    print(f"  {_name}: {len(_od['auc_list'])} entries ({_od['N_STRAPS']} straps × {len(_od['SAMPLE_SIZES'])} sizes)")

## AUC vs sample-size plot

In [ ]:
def _ci95(by_n, samp_list):
    lo, hi = [], []
    for n in samp_list:
        v = np.array(by_n[n])
        lo.append(np.percentile(v, 2.5))
        hi.append(np.percentile(v, 97.5))
    return np.array(lo), np.array(hi)


for _name, _od in outcome_data.items():
    _auc_list = _od["auc_list"]
    if not _auc_list:
        print(f"No AUC data for {_name} — skipping.")
        continue

    _samp_list = np.array(sorted(set(item[1] for item in _auc_list)))
    _mc_by_n   = {n: [item[0][0] for item in _auc_list if item[1] == n] for n in _samp_list}
    _sc_by_n   = {n: [item[0][1] for item in _auc_list if item[1] == n] for n in _samp_list}
    _re_by_n   = {n: [item[0][2] for item in _auc_list if item[1] == n] for n in _samp_list}

    _mc_mean = np.array([np.mean(_mc_by_n[n]) for n in _samp_list])
    _sc_mean = np.array([np.mean(_sc_by_n[n]) for n in _samp_list])
    _re_mean = np.array([np.mean(_re_by_n[n]) for n in _samp_list])

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    for _est, _mean, _by_n in [
        ("Monte Carlo", _mc_mean, _mc_by_n),
        ("SCOPE",       _sc_mean, _sc_by_n),
        ("REACH",       _re_mean, _re_by_n),
    ]:
        _lo, _hi = _ci95(_by_n, _samp_list)
        _c = COLORS[_est]
        ax.plot(_samp_list, _mean, color=_c, label=_est, linewidth=2)
        ax.fill_between(_samp_list, _lo, _hi, color=_c, alpha=0.15)

    ax.set_xlabel("Number of samples per patient")
    ax.set_ylabel("AUC-ROC")
    ax.set_title(f"{_name} — AUC vs samples  (n={_od['n_patients']}, n_pos={_od['n_pos']})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Token-efficiency plot

In [ ]:
def _find_equivalent_samples(target_perf, comp_samples, comp_perf, higher_is_better=True):
    if higher_is_better:
        if target_perf > comp_perf.max():
            return np.nan
        if target_perf <= comp_perf.min():
            return comp_samples[np.argmin(comp_perf)]
    else:
        if target_perf < comp_perf.min():
            return np.nan
        if target_perf >= comp_perf.max():
            return comp_samples[np.argmax(comp_perf)]
    return np.min(comp_samples[comp_perf > target_perf])


def _compute_token_equivalence_curve(mc_samps, mc_perf, comp_samps, comp_perf,
                                     comp_tokens_per_sample, higher_is_better=True):
    equiv_n = np.array([
        _find_equivalent_samples(p, comp_samps, comp_perf, higher_is_better)
        for p in mc_perf
    ])
    return equiv_n * comp_tokens_per_sample


# ── Build per-outcome efficiency data ────────────────────────────────────────
_outcome_eff = {}

for _name, _od in outcome_data.items():
    _auc_list = _od["auc_list"]
    if not _auc_list:
        continue

    _samp_list = np.array(sorted(set(item[1] for item in _auc_list)))
    _mc_by_n   = {n: [item[0][0] for item in _auc_list if item[1] == n] for n in _samp_list}
    _sc_by_n   = {n: [item[0][1] for item in _auc_list if item[1] == n] for n in _samp_list}
    _re_by_n   = {n: [item[0][2] for item in _auc_list if item[1] == n] for n in _samp_list}

    _mc_mean = np.array([np.mean(_mc_by_n[n]) for n in _samp_list])
    _sc_mean = np.array([np.mean(_sc_by_n[n]) for n in _samp_list])
    _re_mean = np.array([np.mean(_re_by_n[n]) for n in _samp_list])

    _avg_m1   = _od["avg_tokens_m1"]
    _avg_m2   = _od["avg_tokens_m2"]
    _avg_reach = _avg_m2 if MARGINAL else (_avg_m1 + _avg_m2)

    _mc_toks = _samp_list * _avg_m1

    # Threshold: min MC tokens needed to beat the worst single-sample estimator
    _worst_single = min(_sc_mean[0], _re_mean[0])
    _above = _mc_toks[_mc_mean >= _worst_single]
    _mc_thresh = float(_above.min()) if len(_above) else np.nan
    _valid_mask = _mc_toks >= (_mc_thresh if not np.isnan(_mc_thresh) else 0)

    _sc_equiv = _compute_token_equivalence_curve(_samp_list, _mc_mean, _samp_list, _sc_mean, _avg_m1)
    _re_equiv = _compute_token_equivalence_curve(_samp_list, _mc_mean, _samp_list, _re_mean, _avg_reach)

    _scope_ratio = np.where(_sc_equiv > 0, _mc_toks / _sc_equiv, np.nan)
    _reach_ratio = np.where(_re_equiv > 0, _mc_toks / _re_equiv, np.nan)

    _outcome_eff[_name] = {
        "mc_toks":     _mc_toks,
        "sc_equiv":    _sc_equiv,
        "re_equiv":    _re_equiv,
        "scope_ratio": _scope_ratio,
        "reach_ratio": _reach_ratio,
        "valid_mask":  _valid_mask,
        "mc_thresh":   _mc_thresh,
    }

if not _outcome_eff:
    print("No efficiency data — skipping combined plot.")
else:
    _n_out   = len(_outcome_eff)
    _cmap    = plt.get_cmap("tab20")
    _palette = [_cmap(i / max(_n_out - 1, 1)) for i in range(_n_out)]

    _reach_label = "REACH (marginal)" if MARGINAL else "REACH (M1+M2)"

    fig, (ax_equiv, ax_ratio) = plt.subplots(1, 2, figsize=(18, 7), dpi=150)

    # Reference lines
    _all_mc_max = max(od["mc_toks"].max() for od in _outcome_eff.values())
    ax_equiv.plot([0, _all_mc_max], [0, _all_mc_max],
                  "k--", alpha=0.4, linewidth=1.5, label="MC reference (y=x)")
    ax_ratio.axhline(y=1.0, color="k", linestyle="--", alpha=0.4,
                     linewidth=1.5, label="Equal efficiency")

    for _color, (_name, _eff) in zip(_palette, _outcome_eff.items()):
        _vm    = _eff["valid_mask"]
        _short = _name.split("//")[-1] if "//" in _name else _name
        ax_equiv.plot(_eff["mc_toks"][_vm], _eff["sc_equiv"][_vm],
                      color=_color, linestyle="-",  linewidth=1.5, label=f"{_short} SCOPE")
        ax_equiv.plot(_eff["mc_toks"][_vm], _eff["re_equiv"][_vm],
                      color=_color, linestyle="--", linewidth=1.5, label=f"{_short} REACH")
        ax_ratio.plot(_eff["mc_toks"][_vm], _eff["scope_ratio"][_vm],
                      color=_color, linestyle="-",  linewidth=1.5, label=f"{_short} SCOPE")
        ax_ratio.plot(_eff["mc_toks"][_vm], _eff["reach_ratio"][_vm],
                      color=_color, linestyle="--", linewidth=1.5, label=f"{_short} REACH")

    ax_equiv.set_xlabel("Monte Carlo Tokens", fontsize=11)
    ax_equiv.set_ylabel("Equivalent Tokens Needed", fontsize=11)
    ax_equiv.set_title(
        "Tokens to Match MC AUROC — All Outcomes\n(solid = SCOPE, dashed = REACH)", fontsize=11
    )
    ax_equiv.set_xlim([0, _all_mc_max * 1.05])
    ax_equiv.set_ylim([0, _all_mc_max * 1.05])
    ax_equiv.grid(True, alpha=0.3)

    ax_ratio.set_xlabel("Monte Carlo Tokens", fontsize=11)
    ax_ratio.set_ylabel("Token Efficiency Ratio (MC Tokens / Estimator Tokens)", fontsize=11)
    ax_ratio.set_title(
        f"Token Efficiency Ratio — All Outcomes\n(solid = SCOPE, dashed = {_reach_label})",
        fontsize=11,
    )
    ax_ratio.grid(True, alpha=0.3)

    _handles, _labels = ax_ratio.get_legend_handles_labels()
    fig.legend(
        _handles, _labels,
        loc="center right", bbox_to_anchor=(1.17, 0.5),
        fontsize=8, ncol=1, framealpha=0.8,
    )
    plt.tight_layout()
    plt.show()

## Calibration curves

In [ ]:
def _calibration_from_edges(targets, predictions, edges):
    bin_idx  = np.clip(np.digitize(predictions, edges) - 1, 0, len(edges) - 2)
    mean_pred = np.zeros(len(edges) - 1)
    frac_pos  = np.zeros(len(edges) - 1)
    mask      = np.zeros(len(edges) - 1, dtype=bool)
    for b in range(len(edges) - 1):
        in_b = bin_idx == b
        if in_b.sum() > 0:
            mean_pred[b] = predictions[in_b].mean()
            frac_pos[b]  = targets[in_b].mean()
            mask[b]      = True
    return frac_pos[mask], mean_pred[mask]


N_BOOT_CAL = 200
rng_cal    = np.random.default_rng(seed=7)

for _name, _od in outcome_data.items():
    _y      = _od["y_true"]
    _M0, _M1, _M2 = _od["M0"], _od["M1"], _od["M2"]
    _n_pts  = _od["n_patients"]
    _N_BINS = _od["N_BINS"]

    _all_preds = np.concatenate([_M0[~np.isnan(_M0)], _M1[~np.isnan(_M1)], _M2[~np.isnan(_M2)]])
    if len(_all_preds) == 0:
        print(f"No predictions for {_name} — skipping calibration.")
        continue

    _edges       = np.percentile(_all_preds, np.linspace(0, 100, _N_BINS + 1))
    _edges[0]   -= 1e-9
    _edges[-1]  += 1e-9

    fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5, label="Perfect")

    for _est, _scores in [("Monte Carlo", _M0), ("SCOPE", _M1), ("REACH", _M2)]:
        _valid = ~np.isnan(_scores)
        _preds = np.clip(_scores[_valid], 0, 1)
        _tgts  = _y[_valid].astype(float)
        _c     = COLORS[_est]

        _boot_curves = []
        _n = len(_preds)
        for _ in range(N_BOOT_CAL):
            _idx = rng_cal.integers(0, _n, size=_n)
            _fp, _mp = _calibration_from_edges(_tgts[_idx], _preds[_idx], _edges)
            if len(_mp) > 1:
                _boot_curves.append(np.interp(np.linspace(0, 1, 100), _mp, _fp))
        if _boot_curves:
            _bc = np.array(_boot_curves)
            _xg = np.linspace(0, 1, 100)
            ax.fill_between(_xg, np.percentile(_bc, 2.5, axis=0),
                            np.percentile(_bc, 97.5, axis=0), color=_c, alpha=0.15)

        _fp_full, _mp_full = _calibration_from_edges(_tgts, _preds, _edges)
        _bs = brier_score_loss(_tgts, _preds)
        ax.plot(_mp_full, _fp_full, color=_c, marker="o", linewidth=2,
                label=f"{_est}  (Brier={_bs:.4f})")

    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction positive")
    ax.set_title(f"{_name} — Calibration  (n={_n_pts}, {_N_BINS} bins)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()